# Analyze Time-Warp

Analyzing TWarp with No Fine-Tuning

In [ ]:
import numpy as np
import pandas as pd
import yaml
import time
import sys
import os
import os.path as osp
import joblib
import pickle
from itertools import product
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
# Local modules
from src.utils import read_yml, Dict, time_range, time_intp, plot_styles
from src.models import moisture_rnn as mrnn

In [ ]:
# document-safe plotting defaults
FIGSIZE = (6.5,4.5)
DPI = 300
LABEL_SIZE = 14
TICK_SIZE = 12
CBAR_LABEL_SIZE = 13

In [ ]:
# Utilty func to read text file summaries
def read_rep_report(path):
    path = Path(path)
    metrics, meta = {}, {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            if line.startswith("Median replication index"):
                meta["median_replication_index"] = int(line.split(":")[1])
            elif line.startswith("Seed directory"):
                meta["seed_directory"] = line.split(":",1)[1].strip()
            elif line.startswith("Twarp Params"):
                meta["twarp_params"] = line.split(":",1)[1].strip()
            elif ":" in line:
                k,v = line.split(":",1)
                try: metrics[k.strip()] = float(v)
                except ValueError: pass
    return {**meta, **metrics}

## Data

In [ ]:
reps_dir = "outputs/transfer0_reps"

wtrain = pd.read_csv(osp.join("outputs/transfer0/weather_train.csv"))
wtest  = pd.read_csv(osp.join("outputs/transfer0/weather_test.csv"))

In [ ]:
# Summaries with Median Accuracy across Reps
fm1_med_meta    = read_rep_report(osp.join(reps_dir, "fm1_median_rep_report.txt"))
fm100_med_meta  = read_rep_report(osp.join(reps_dir, "fm100_median_rep_report.txt"))
fm1000_med_meta = read_rep_report(osp.join(reps_dir, "fm1000_median_rep_report.txt"))

In [ ]:
# All Model Reps
results = [
    pd.read_pickle(p / "results_test_set.pkl")
    for p in sorted(
        (p for p in Path(reps_dir).iterdir() if p.is_dir() and p.name.startswith("seed_")),
        key=lambda p: int(p.name.split("_")[1])
    )
]

## Check Error relationship within reps

We want to use the median accuracy case for each FMC class. We check the relationship between accuracy in each rep for the different fuel classes. We see modest positive correlation.

In [ ]:
fm1_rmse_30 = np.array([r["FM1"]["rmse_30"] for r in results])
fm100_rmse  = np.array([r["FM100"]["rmse"] for r in results])
fm1000_rmse = np.array([r["FM1000"]["rmse"] for r in results])

In [ ]:
np.corrcoef(np.vstack([fm1_rmse_30, fm100_rmse, fm1000_rmse]))

## Extract Models with Median Accuracy from Replications

We briefly check the accuracy for the other fuel classes for the median rep, to make sure the median case isn't biased towards poor preforming other classes

In [ ]:
fm1_med    = results[fm1_med_meta["median_replication_index"]]["FM1"]
fm100_med  = results[fm100_med_meta["median_replication_index"]]["FM100"]
fm1000_med = results[fm1000_med_meta["median_replication_index"]]["FM1000"]

In [ ]:
print(f'{fm1_med_meta["median_replication_index"]=}')
print(f'{fm100_med_meta["median_replication_index"]=}')
print(f'{fm1000_med_meta["median_replication_index"]=}')

In [ ]:
print("FM1 Median Accuracy Case")
print(f"FM1 RMSE (<30): {fm1_med['rmse_30']}")
print(f'FM100 RMSE: {results[fm1_med_meta["median_replication_index"]]["FM100"]["rmse"]}')
print(f'FM1000 RMSE: {results[fm1_med_meta["median_replication_index"]]["FM1000"]["rmse"]}')

In [ ]:
print("FM100 Median Accuracy Case")
print(f'FM1 RMSE (<30): {results[fm100_med_meta["median_replication_index"]]["FM1"]["rmse_30"]}')
print(f'FM100 RMSE: {fm100_med["rmse"]}')
print(f'FM1000 RMSE: {results[fm100_med_meta["median_replication_index"]]["FM1000"]["rmse"]}')

Replication 23 had the median RMSE for both FM100 and FM1000, and the associated FM1 was slightly less than the estimated median FM1 RMSE_30. Thus, we will use replication 23 as the individual case to genenerate summary plots.

### FM1 Results

In [ ]:
fm1 = results[fm100_med_meta["median_replication_index"]]["FM1"]

In [ ]:
keys = ["rmse", "bias", "r2", "rmse_30", "bias_30", "r2_30"]
print(f"FM1 Median Accuracy Results")
print(f"Time-Warp Params: {fm1['params']}")
for k in keys:
    print(f"    {k}: {np.round(fm1[k], 4)}")

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.scatter(fm1['fm1_obs'], fm1['preds1_intp'], s=50, alpha=0.8, facecolor=plot_styles["model1"]["color"], edgecolor="k", linewidth=0.5, zorder=3)
ax.set_xlabel("Observed FM1 (%)", fontsize=LABEL_SIZE)
ax.set_ylabel("Predicted FM1 (%)", fontsize=LABEL_SIZE)
# Compute common limits
xmin = min(fm1['fm1_obs'].min(), fm1['preds1_intp'].min())
xmax = max(fm1['fm1_obs'].max(), fm1['preds1_intp'].max())
ax.plot([xmin, xmax], [xmin, xmax],
        linestyle="dashed", color="k", zorder=2, label="y = x")
# ax.set_title("All FM10 Observations", fontsize=LABEL_SIZE)
ax.tick_params(labelsize=TICK_SIZE)
ax.text(
    0.05, 0.95,
    r"$R^2$ = {:.2f}".format(fm1["r2"]),
    transform=ax.transAxes,
    fontsize=LABEL_SIZE,
    verticalalignment="top",
    color="black"
)
ax.grid()
plt.tight_layout()
plt.savefig("outputs/fm1_scatter.png")

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
inds = np.where(fm1['fm1_obs'] < 30)
ax.scatter(fm1['fm1_obs'][inds], fm1['preds1_intp'][inds], s=50, alpha=0.8, facecolor=plot_styles["model1"]["color"], edgecolor="k", linewidth=0.5, zorder=3)
ax.set_xlabel("Observed FM1 (%)", fontsize=LABEL_SIZE)
ax.set_ylabel("Predicted FM1 (%)", fontsize=LABEL_SIZE)
# Compute common limits
xmin = min(fm1['fm1_obs'][inds].min(), fm1['preds1_intp'][inds].min())
xmax = max(fm1['fm1_obs'][inds].max(), fm1['preds1_intp'][inds].max())
ax.plot([xmin, xmax], [xmin, xmax],
        linestyle="dashed", color="k", zorder=2, label="y = x")
ax.tick_params(labelsize=TICK_SIZE)
# ax.set_title(r"Observed FM1$\leq 30\%$)", fontsize=LABEL_SIZE)
ax.text(
    0.05, 0.95,
    r"$R^2$ = {:.2f}".format(fm1["r2_30"]),
    transform=ax.transAxes,
    fontsize=LABEL_SIZE,
    verticalalignment="top",
    color="black"
)
ax.set_ylim(0, 30)
ax.grid()
plt.tight_layout()
plt.savefig("outputs/fm1_30_scatter.png")

In [ ]:
print(f"N Total: {fm1['fm1_obs'].shape[0]}")
print(f"N <=30: {fm1['fm1_obs'][inds].shape[0]}")

In [ ]:
plt_start = pd.Timestamp('1997-08-13 14:00:00')
plt_end   = plt_start + pd.DateOffset(hours=72)
inds = np.where((fm1["times"] >= plt_start) & (fm1["times"] <= plt_end))
inds2 = np.where((fm1["times_fm1"] >= plt_start) & (fm1["times_fm1"] <= plt_end))

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(fm1['times'].iloc[inds], fm1['preds1'][inds], **plot_styles["model1"], 
        linewidth=2.5)
# ax.scatter(fm1['times_fm1'].iloc[inds2], fm1['fm1_obs'][inds2], **plot_styles["fm1"], edgecolor="k")
ax.scatter(fm1['times_fm1'].iloc[inds2], fm1['fm1_obs'][inds2], **plot_styles["fm1"], s=50, edgecolor="k", linewidths=0.5)
ax.tick_params(axis='x', labelrotation=45, labelsize=TICK_SIZE)
ax.set_ylabel("FMC (%)", fontsize=LABEL_SIZE)
ax.grid()
fig.legend(loc='upper left', bbox_to_anchor=(.9, .8), fontsize=CBAR_LABEL_SIZE)

## FM100 Results

In [ ]:
fm100 = results[fm100_med_meta["median_replication_index"]]["FM100"]

In [ ]:
keys = ["rmse", "bias", "r2"]
print(f"FM100 Results")
print(f"Time-Warp Params: {fm100['params']}")
for k in keys:
    print(f"    {k}: {np.round(fm100[k], 4)}")

In [ ]:
print(f"N Total: {fm100['fm100_obs'].shape[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
# ax.scatter(fm100['fm100_obs'], fm100['preds100_intp'], **plot_styles["model100"])
ax.scatter(fm100['fm100_obs'], fm100['preds100_intp'], s=50, alpha=0.8, facecolor=plot_styles["model100"]["color"], edgecolor="k", linewidth=0.5, zorder=3)
ax.set_xlabel("Observed FM100 (%)", fontsize=LABEL_SIZE)
ax.set_ylabel("Predicted FM100 (%)", fontsize=LABEL_SIZE)
# Compute common limits
xmin = min(fm100['fm100_obs'].min(), fm100['preds100_intp'].min())
xmax = max(fm100['fm100_obs'].max(), fm100['preds100_intp'].max())
ax.plot([xmin, xmax], [xmin, xmax],
        linestyle="dashed", color="k", zorder=2, label="y = x")
ax.text(
    0.05, 0.95,
    r"$R^2$ = {:.2f}".format(fm100["r2"]),
    transform=ax.transAxes,
    fontsize=LABEL_SIZE,
    verticalalignment="top",
    color="black"
)
ax.set_ylim(0, 30)
ax.tick_params(labelsize=TICK_SIZE)
ax.grid()
plt.tight_layout()
plt.savefig("outputs/fm100_scatter.png")

In [ ]:
plt_start = pd.Timestamp('1997-08-13 14:00:00')
plt_end   = plt_start + pd.DateOffset(hours=72)
inds = np.where((fm100["times"] >= plt_start) & (fm100["times"] <= plt_end))
inds2 = np.where((fm100["times_fm100"] >= plt_start) & (fm100["times_fm100"] <= plt_end))

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(fm100['times'].iloc[inds], fm100['preds100'][inds], **plot_styles["model100"])
ax.scatter(fm100['times_fm100'].iloc[inds2], fm100['fm100_obs'][inds2], **plot_styles["fm100"])

ax.tick_params(axis='x', labelrotation=45, labelsize=TICK_SIZE)
ax.set_ylabel("FMC (%)", fontsize=LABEL_SIZE)
ax.grid()
fig.legend(loc='upper left', bbox_to_anchor=(.7, 1), fontsize=CBAR_LABEL_SIZE)

## FM1000 Results

In [ ]:
fm1000 = results[fm1000_med_meta["median_replication_index"]]["FM1000"]

In [ ]:
print(f"N Total: {fm1000['fm1000_obs'].shape[0]}")

In [ ]:
keys = ["rmse", "bias", "r2"]
print(f"FM1000 Results")
print(f"Time-Warp Params: {fm1000['params']}")
for k in keys:
    print(f"    {k}: {np.round(fm1000[k], 4)}")

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
# ax.scatter(fm1000['fm1000_obs'], fm1000['preds1000_intp'], **plot_styles["model1000"])
ax.scatter(fm1000['fm1000_obs'], fm1000['preds1000_intp'], s=50, alpha=0.8, facecolor=plot_styles["model1000"]["color"], edgecolor="k", linewidth=0.5, zorder=3)
ax.set_xlabel("Observed FM1000 (%)", fontsize=LABEL_SIZE)
ax.set_ylabel("Predicted FM1000 (%)", fontsize=LABEL_SIZE)
# Compute common limits
xmin = min(fm1000['fm1000_obs'].min(), fm1000['preds1000_intp'].min())
xmax = max(fm1000['fm1000_obs'].max(), fm1000['preds1000_intp'].max())
ax.plot([xmin, xmax], [xmin, xmax],
        linestyle="dashed", color="k", zorder=2, label="y = x")
ax.text(
    0.05, 0.95,
    r"$R^2$ = {:.2f}".format(fm1000["r2"]),
    transform=ax.transAxes,
    fontsize=LABEL_SIZE,
    verticalalignment="top",
    color="black"
)
ax.set_ylim(0, 30)
ax.tick_params(labelsize=TICK_SIZE)
ax.grid()
plt.tight_layout()
plt.savefig("outputs/fm1000_scatter.png")

In [ ]:
plt_start = pd.Timestamp('1997-08-13 12:00:00')
plt_end   = plt_start + pd.DateOffset(hours=72)
inds = np.where((fm1000["times"] >= plt_start) & (fm1000["times"] <= plt_end))
inds2 = np.where((fm1000["times_fm1000"] >= plt_start) & (fm1000["times_fm1000"] <= plt_end))

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(fm1000['times'].iloc[inds], fm1000['preds1000'][inds], **plot_styles["model1000"])
ax.scatter(fm1000['times_fm1000'].iloc[inds2], fm1000['fm1000_obs'][inds2], **plot_styles["fm1000"])

ax.tick_params(axis='x', labelrotation=45, labelsize=TICK_SIZE)
ax.set_ylabel("FMC (%)", fontsize=LABEL_SIZE)
ax.grid()
fig.legend(loc='upper left', bbox_to_anchor=(.7, 1), fontsize=CBAR_LABEL_SIZE)